In [2]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 13.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 13.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd

df = pd.read_csv('ALA_S12026PE1.csv')
print(f"Original shape: {df.shape}")

Original shape: (25276, 11)


In [12]:
# Confirm the records before removing
invalid_coords = df[df['decimalLongitude'] > 154]
print("Records to remove:")
print(invalid_coords[['observationDate', 'stateProvince', 
                        'decimalLatitude', 'decimalLongitude', 
                        'basisOfRecord', 'dataResourceName']])

# Remove them
df = df[df['decimalLongitude'] <= 154]
print(f"Shape after Fix 1: {df.shape}")
# Should be 25273 rows (removed 3)

Records to remove:
      observationDate    stateProvince  decimalLatitude  decimalLongitude  \
190          1/9/1968  New South Wales       -31.580000        159.080000   
19715        1/9/1968  New South Wales       -31.538675        159.080004   
20700       24/5/2020  New South Wales       -31.525040        159.067610   

           basisOfRecord                dataResourceName  
190    HUMAN_OBSERVATION  New South Wales Bird Atlassers  
19715  HUMAN_OBSERVATION                NSW BioNet Atlas  
20700  HUMAN_OBSERVATION                NSW BioNet Atlas  
Shape after Fix 1: (25273, 11)


In [13]:
# Confirm the record before fixing
mismatch = df[(df['dataResourceUid'] == 'dr1411') & 
              (df['dataResourceName'] == 'eBird Australia')]
print("Record to fix:")
print(mismatch[['dataResourceUid', 'dataResourceName', 
                'observationDate', 'stateProvince']])

# Fix it
df.loc[(df['dataResourceUid'] == 'dr1411') & 
       (df['dataResourceName'] == 'eBird Australia'), 
       'dataResourceName'] = 'iNaturalist Australia'

# Verify
print("dr1411 names after fix:")
print(df[df['dataResourceUid'] == 'dr1411']['dataResourceName'].value_counts())
# Should show only iNaturalist Australia: 587

Record to fix:
     dataResourceUid dataResourceName observationDate stateProvince
5404          dr1411  eBird Australia       10/8/2014      Victoria
dr1411 names after fix:
dataResourceName
iNaturalist Australia    587
Name: count, dtype: int64


In [14]:
# ── Fix 3: Correct all wrong stateProvince labels ──

# 3a. NSW-labelled record whose coordinates are in Tasmania
wrong_nsw = df[(df['stateProvince'] == 'New South Wales') & 
               (df['decimalLatitude'] < -38)]
print("NSW record with Tasmanian coordinates:")
print(wrong_nsw[['observationDate', 'stateProvince', 
                  'decimalLatitude', 'decimalLongitude', 
                  'dataResourceName', 'recordID']])

df.loc[df['recordID'] == '6a621dc2-f4a0-4e93-9b54-9520a650da33', 
       'stateProvince'] = 'Tasmania'

# 3b. ACT-labelled records whose coordinates are in NSW
wrong_act = df[(df['stateProvince'] == 'Australian Capital Territory') & 
               (df['decimalLongitude'] > 149.4)]
print("\nACT records with NSW coordinates:")
print(wrong_act[['observationDate', 'stateProvince', 
                  'decimalLatitude', 'decimalLongitude', 
                  'dataResourceName', 'recordID']])

act_wrong_ids = wrong_act['recordID'].tolist()
df.loc[df['recordID'].isin(act_wrong_ids), 
       'stateProvince'] = 'New South Wales'

# Verify both fixes
print("\nVerification - NSW record now Tasmania:")
print(df[df['recordID'] == '6a621dc2-f4a0-4e93-9b54-9520a650da33']
        [['stateProvince', 'decimalLatitude', 'decimalLongitude']])

print("\nVerification - ACT records now NSW:")
print(df[df['recordID'].isin(act_wrong_ids)]
        [['stateProvince', 'decimalLongitude']])

NSW record with Tasmanian coordinates:
      observationDate    stateProvince  decimalLatitude  decimalLongitude  \
22722      26/10/1977  New South Wales       -42.081816        148.417987   

                     dataResourceName                              recordID  
22722  Tasmanian Natural Values Atlas  6a621dc2-f4a0-4e93-9b54-9520a650da33  

ACT records with NSW coordinates:
      observationDate                 stateProvince  decimalLatitude  \
6095        24/4/2017  Australian Capital Territory            -35.1   
18992        6/5/2009  Australian Capital Territory            -35.2   
19814       24/4/2017  Australian Capital Territory            -35.1   

       decimalLongitude  dataResourceName  \
6095              150.7   eBird Australia   
18992             150.8  NSW BioNet Atlas   
19814             150.7  NSW BioNet Atlas   

                                   recordID  
6095   7513d487-460a-45d1-854e-6ca4e38795ab  
18992  c7c5b708-e00a-4316-bf43-87957992a177  
19814  

In [15]:
df.to_csv('ALA_S12026PE1_cleaned.csv', index=False)
print(f"Cleaned file saved. Final shape: {df.shape}")

Cleaned file saved. Final shape: (25273, 11)
